In [10]:
import os
import re
import json
import fitz 
import asyncio
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from types import SimpleNamespace

load_dotenv()

True

In [2]:
def process_filename(file_path: str | Path, limit_folder=None) -> str:
    path_obj = Path(file_path).resolve()
    return f"\\\\?\\{path_obj}"

def parse_pdf_text(filepath: str):
    safe_path = process_filename(filepath)
    
    try:
        with open(safe_path, "rb") as f:
            file_bytes = f.read()
            
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        
        pages_list = []
        for page in doc:
            text_content = page.get_text("text") 
            
            page_obj = SimpleNamespace(markdown=text_content)
            pages_list.append(page_obj)
        
        return SimpleNamespace(pages=pages_list)
        
    except OSError as e:
        print(f"Error opening {Path(filepath).name}: {e}")
        return None

def parse_pdf(filepath: str, method: str = "text"):
    if method == "ocr":
        raise NotImplementedError("OCR method is not implemented yet")
    elif method == "text":
        return parse_pdf_text(filepath)
    else:
        raise ValueError("Method must be 'ocr' or 'text'")


In [5]:
filename_to_len = {}
base_dir = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\cleaned_downloads"
pdf_files = glob(os.path.join(base_dir, "*.pdf"))

for filepath in tqdm(pdf_files, desc="Processing PDFs"):
    response = parse_pdf(filepath, method="text")
    if response is not None:
        total_length = sum(len(page.markdown) for page in response.pages)
        filename_to_len[Path(filepath).name] = total_length

Processing PDFs:   0%|          | 0/2456 [00:00<?, ?it/s]

Processing PDFs:  13%|█▎        | 317/2456 [00:35<11:44,  3.03it/s]

MuPDF error: format error: No default Layer config



Processing PDFs:  99%|█████████▊| 2423/2456 [05:19<00:09,  3.31it/s]

MuPDF error: format error: No default Layer config



...

Processing PDFs:  99%|█████████▉| 2428/2456 [05:21<00:08,  3.19it/s]

MuPDF error: format error: No default Layer config



Processing PDFs:  99%|█████████▉| 2429/2456 [05:21<00:08,  3.31it/s]

MuPDF error: format error: No default Layer config



Processing PDFs: 100%|██████████| 2456/2456 [05:26<00:00,  7.52it/s]


In [7]:
import asyncio
import aiohttp
import time

# Update this to match your vLLM server settings
VLLM_URL = "http://localhost:8000/v1/chat/completions"
MODEL_NAME = "QuantTrio/Qwen3.5-27B-AWQ"

async def async_call_qwen(session: aiohttp.ClientSession, prompt: str, max_tokens: int = 1500) -> str:
    """Async helper to call the local vLLM server."""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a highly capable legal AI assistant. You must write all of your responses entirely in Bahasa Indonesia."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,
        "temperature": 0.1 # Low temperature for factual extraction
    }
    
    async with session.post(VLLM_URL, json=payload) as response:
        response.raise_for_status()
        result = await response.json()
        return result["choices"][0]["message"]["content"]


async def process_worker_chunk(session: aiohttp.ClientSession, chunk_id: int, chunk_text: str, blueprint: str, sem: asyncio.Semaphore) -> str:
    """
    PHASE 2: The Worker (With Discovery Clause)
    Evidence: arXiv:2509.20900 (Asymmetric Agents) & Hybrid Extraction.
    Forces strict schema compliance to prevent token waste, but utilizes a "catch-all" 
    discovery clause to prevent Inattentional Blindness to critical out-of-schema facts.
    """
    worker_prompt = f"""
    You are a precise legal assistant reading chunk #{chunk_id} of a Constitutional Court decision. 

    TASK 1 (Blueprint Extraction):
    Extract facts from this chunk strictly according to this Blueprint:
    {blueprint}

    TASK 2 (Discovery Clause):
    If you discover a highly critical legal argument or fact in this chunk that is ESSENTIAL 
    but DOES NOT fit into the Blueprint, extract it under "### Temuan Kritis Tambahan".

    If the chunk contains nothing relevant to the blueprint or the discovery clause, output exactly: "Tidak ada data relevan yang ditemukan."

    CRITICAL: All extracted facts, notes, and headings MUST be written entirely in formal Bahasa Indonesia. Do not translate Indonesian legal terms into English.

    Chunk text: {chunk_text}
    """
    
    # The semaphore ensures only 3 chunks are processed at the exact same time
    async with sem:
        print(f"⏳ [Worker {chunk_id}] Sending to vLLM...")
        start_time = time.time()
        
        extraction = await async_call_qwen(session, worker_prompt, max_tokens=3000)
        
        elapsed = time.time() - start_time
        print(f"✅ [Worker {chunk_id}] Finished in {elapsed:.2f} seconds.")
        
        return f"\n--- CHUNK {chunk_id} EXTRACTIONS ---\n{extraction}"


async def run_summarization_pipeline(pages: list[str], chunk_size: int = 10):
    print("🚀 Starting Async Summarization Pipeline...")
    overall_start = time.time()
    
    async with aiohttp.ClientSession() as session:
        
        # ==========================================
        # PHASE 1: The Planner
        # Evidence: arXiv:2512.17179 (Self-Planning)
        # Bypasses the "Lost in the Middle" phenomenon by reading only the bookends 
        # to establish a global structural blueprint before processing the deep text.
        # ==========================================
        print("\n📝 [Phase 1] Generating Blueprint...")
        planner_context = "\n".join(pages[:3] + pages[-3:])
        blueprint_prompt = f"""
        You are a senior legal analyst. Based on the beginning and end of this court document, 
        create a 4-point structural blueprint of the most critical information needed for a summary.

        CRITICAL: The blueprint MUST be written in Bahasa Indonesia.
        Respond ONLY with the bulleted blueprint.
        Document text: {planner_context}
        """
        blueprint = await async_call_qwen(session, blueprint_prompt, max_tokens=1500)
        print("✅ Blueprint Generated:\n", blueprint)


        # ==========================================
        # PHASE 2: The Worker (Map)
        # Process chunks concurrently with a strict concurrency limit of 3.
        # ==========================================
        print(f"\n⚙️ [Phase 2] Spawning Workers (Batch Size: 3)...")
        sem = asyncio.Semaphore(3) # Limits concurrent vLLM calls to 3
        tasks = []
        
        # Slice the pages into chunks
        chunks = ["\n".join(pages[i : i + chunk_size]) for i in range(0, len(pages), chunk_size)]
        
        for idx, chunk_text in enumerate(chunks):
            # Create a task for each chunk, but the semaphore will throttle execution
            task = asyncio.create_task(process_worker_chunk(session, idx + 1, chunk_text, blueprint, sem))
            tasks.append(task)
            
        # Wait for all chunks to finish processing
        worker_results = await asyncio.gather(*tasks)
        combined_extractions = "\n".join(worker_results)


        # ==========================================
        # PHASE 3: The Synthesizer (Adaptive Merge)
        # Evidence: arXiv:2502.00977 (Context-Aware Hierarchical Merging)
        # Prevents hallucination amplification by merging extracted evidence (not raw text), 
        # and explicitly grounds the model by notifying it of out-of-schema discoveries.
        # ==========================================
        print("\n⚖️ [Phase 3] Synthesizing Final Summary...")
        synthesizer_prompt = f"""
        You are an expert Chief Justice synthesizing notes from an 80-page decision. 

        The notes are organized by this original Blueprint: 
        {blueprint}

        However, reviewers also flagged "Additional Critical Findings". 
        Synthesize ALL notes into a comprehensive executive summary. Weave the unexpected 
        findings logically into the narrative. Eliminate duplicates.

        CRITICAL: The final executive summary MUST be written in highly professional, formal Bahasa Indonesia suitable for a legal context.

        Extracted Notes:
        {combined_extractions}
        """
        
        final_summary = await async_call_qwen(session, synthesizer_prompt, max_tokens=2500)
        
        total_time = time.time() - overall_start
        print(f"\n🎉 Pipeline Complete in {total_time:.2f} seconds!")
        print("\n================ FINAL SUMMARY ================\n")
        print(final_summary)
        
        return final_summary

In [ ]:

# --- How to Execute ---
# Assuming 'response.pages' contains your parsed PDF
# pages_list = [page.markdown for page in response.pages]
# final_output = asyncio.run(run_summarization_pipeline(pages_list))

In [118]:
import os
import asyncio
from openai import AsyncOpenAI

async def async_call_llm(client: AsyncOpenAI, model_name: str, prompt: str, max_tokens: int = 1500) -> str:
    response = await client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system", 
                "content": """You are a highly capable legal Data Engineer.
CRITICAL RULES:
1. BAHASA: Gunakan Bahasa Indonesia formal yang lugas. 
2. ANTI-BELANDA/LATIN: DILARANG KERAS menggunakan istilah seperti "Verstek", "Niet Ontvankelijke", "Obscuur Libel", "Wanprestasi", "Ex Aequo Et Bono", dsb. 
   - Gunakan: "Putusan tanpa kehadiran", "Gugatan tidak dapat diterima", "Gugatan kabur", "Ingkar janji".
3. ANONIMISASI: Gunakan [Penggugat], [Tergugat], [Objek Sengketa], [Nomor Perkara].
4. FOKUS REGULASI: Prioritaskan interpretasi fakta berdasarkan pasal-pasal di KUHPerdata.
                """
            },
            {"role": "user", "content": prompt}
        ],
        max_completion_tokens=max_tokens,
        temperature=0.1 
    )
    return response.choices[0].message.content

async def process_worker_chunk(client: AsyncOpenAI, model_name: str, chunk_id: int, chunk_text: str, blueprint: str, sem: asyncio.Semaphore) -> str:
    worker_prompt = f"""
    TUGAS: Anda adalah analis hukum yang sangat teliti. Anda sedang memeriksa BAGIAN #{chunk_id} dari sebuah dokumen putusan.

    KONTEKS GLOBAL (Blueprint sebagai Panduan):
    {blueprint}

    INSTRUKSI KHUSUS EKSTRAKSI:
    1. OBSERVASI LOKAL: Hanya ekstrak informasi yang BENAR-BENAR MUNCUL dalam teks Chunk #{chunk_id} di bawah ini. Jangan mengulang informasi dari Blueprint jika tidak ada buktinya di teks chunk ini.
    2. IDENTIFIKASI TAHAPAN: Tentukan apakah chunk ini berisi: 'Identitas Para Pihak', 'Duduk Perkara (Kronologi)', 'Pertimbangan Hukum (Ratio Decidendi)', atau 'Amar Putusan'. Fokuskan ekstraksi pada fungsi bagian tersebut.
    3. PENERAPAN GROUND TRUTH: Jika ada pasal yang disebutkan, jangan hanya menyalin isinya. Jelaskan: "Hakim menggunakan pasal ini UNTUK menilai [fakta apa]".

    FORMAT OUTPUT WAJIB:
    ### [A] Analisis Chunk #{chunk_id}
    - **Kategori Dokumen**: (Sebutkan: Identitas/Duduk Perkara/Pertimbangan/Amar)
    - **Detail Unik**: (Ekstrak fakta spesifik atau argumen yang hanya ada di chunk ini)
    - **Logika Hukum**: (Bagaimana pihak atau hakim membangun argumen di bagian ini)

    ### [B] Temuan Regulasi Spesifik (Jika Ada)
    - **Pasal/Doktrin**: [Nomor Pasal]
    - **Konteks di Chunk**: [Mengapa pasal ini muncul di sini?]

    Dilarang mengeluarkan output "Tidak ada data relevan" kecuali chunk tersebut benar-benar kosong atau hanya berisi disclaimer teknis.

    TEKS CHUNK #{chunk_id}:
    {chunk_text}
    """
    async with sem:
        extraction = await async_call_llm(client, model_name, worker_prompt, max_tokens=3000)
        print(f"✅ [Worker {chunk_id}] Finished.")
        return f"\n--- CHUNK {chunk_id} EXTRACTIONS ---\n{extraction}"


async def run_summarization_pipeline(
        client: AsyncOpenAI, 
        model_name: str, 
        pages: list[str], 
        # chunk_size: int = 10, 
        worker_batch_size: int = 3,
        chunk_size_words: int = 5000,
        overlap_words: int = 200
    ):
    print(f"🚀 Starting Pipeline using model: {model_name}...")
    full_text = " ".join(pages)
    words = full_text.split()
    
    # The Planner
    # Evidence: arXiv:2512.17179 (Self-Planning)
    # Bypasses the "Lost in the Middle" phenomenon by reading only the bookends 
    # to establish a global structural blueprint before processing the deep text.
    print("\n📝 [Phase 1] Generating Blueprint...")
    planner_context = "\n".join(pages[:5] + pages[-5:])
    blueprint_prompt = f"""
    Identifikasi 3 elemen dari dokumen ini:
    1. Subjek & Objek: Siapa [Penggugat], [Tergugat], dan apa [Objek Sengketa]?
    2. Garis Besar Konflik: Inti masalah (misal: jual beli, utang piutang).
    3. Hasil Akhir: Apakah gugatan dikabulkan, ditolak, atau tidak dapat diterima?

    Gunakan informasi ini sebagai konteks untuk proses ekstraksi detail nantinya.
    
    Kembalikan blueprint dalam format poin-poin yang jelas dan terstruktur.
    Document text: {planner_context}
    """
    blueprint = await async_call_llm(client, model_name, blueprint_prompt, max_tokens=1500)
    print("✅ Blueprint Generated:\n", blueprint)
    
    # The Worker 
    # Evidence: Zero-Shot Open-Schema Entity Structure Discovery, Arxiv
    # Forces strict schema compliance to prevent token waste, but utilizes a "catch-all" 
    # discovery clause to prevent Inattentional Blindness to critical out-of-schema facts.
    print(f"\n[Phase 2] Spawning Workers (Batch Size: {worker_batch_size})...")
    chunks = []
    # Logika Overlapping: i maju sebanyak (chunk_size - overlap)
    step = chunk_size_words - overlap_words
    for i in range(0, len(words), step):
        chunk_content = " ".join(words[i : i + chunk_size_words])
        chunks.append(chunk_content)
    
    sem = asyncio.Semaphore(worker_batch_size)
    tasks = [
        asyncio.create_task(process_worker_chunk(client, model_name, idx + 1, chunk_text, blueprint, sem))
        for idx, chunk_text in enumerate(chunks)
    ]
    worker_results = await asyncio.gather(*tasks)
    combined_extractions = "\n".join(worker_results)

    # The Synthesizer (Adaptive Merge)
    # Evidence: arXiv:2502.00977 (Context-Aware Hierarchical Merging)
    # Prevents hallucination amplification by merging extracted evidence (not raw text), 
    # and explicitly grounds the model by notifying it of out-of-schema discoveries.
    print("\n[Phase 3] Synthesizing Final Summary...")
    synthesizer_prompt = f"""
Anda adalah Senior Legal Data Engineer. Ubah catatan ekstraksi menjadi query yang relevan.

ATURAN KHUSUS:
1. HAPUS SEMUA ISTILAH BELANDA/LATIN. Jika ada "Verstek", ubah jadi "Putusan tanpa kehadiran". Jika ada "Obscuur Libel", ubah jadi "Gugatan tidak jelas/kabur". Jika ada "NO", ubah jadi "Gugatan tidak dapat diterima".
2. FOKUS KUHPERDATA: Hubungkan temuan fakta dengan pasal-pasal di KUHPerdata 
3. BAHASA: jangan gunakan akronim atau istilah yang tidak umum. Gunakan Bahasa Indonesia sebagai abstraksi dari istilah pada notes.

PENTING: 
- Pada bagian [Relevansi], HANYA masukkan pasal yang benar-benar dibahas atau menjadi dasar logika di dalam kalimat query tersebut. Jangan memasukkan semua pasal jika query hanya membahas satu aspek spesifik. Jika query membahas prosedur, jangan masukkan pasal materiil.
- Jangan melakukan over-labeling. Jika sebuah query membahas tentang kegagalan prosedur atau kejelasan objek, jangan mencantumkan pasal materiil kecuali query tersebut secara eksplisit menanyakan tentang unsur kesalahan atau kerugian.

STRUKTUR OUTPUT:

1. Query Teknis: Satu kalimat tanya yang menggabungkan cacat prosedur (gugatan kabur) dengan inti masalah (misal: sengketa jual beli). 
   WAJIB [Relevansi]: [Relevansi: Pasal XXX KUHPerdata, ...].

2. Pertanyaan Hukum Orang Awam: Satu pertanyaan natural dari perspektif non legal. Contoh: "Kenapa saya kalah padahal lawan tidak datang sidang?" 
   WAJIB [Relevansi]: [Relevansi: Pasal XXX KUHPerdata, ...].

3. Query Anatomi Fakta (Pola): [Jenis Perjanjian] + [Pelanggaran] + [Alasan Hakim Menolak]. Abstraksikan 3 pola menjadi kalimat gabunganyang mudah dipahami tanpa notasi "+".
   WAJIB [Relevansi]: [Relevansi: Pasal XXX KUHPerdata, ...].

4. Ringkasan Kasus: Maksimal 5 kalimat. Fokus pada: Fakta perikatan, tindakan ingkar janji, dan alasan logis hakim menyatakan gugatan kabur karena tuntutan yang saling bertentangan.
    WAJIB [Relevansi]: [Relevansi: Pasal XXX KUHPerdata, dst].

5. Daftar Pasal yang Diuji: List pasal KUHPerdata saja beserta substansinya (maksimal 15 kata). 

Extracted Notes:
{combined_extractions}
"""

    final_summary = await async_call_llm(client, model_name, synthesizer_prompt, max_tokens=2500)
    print("\n================ FINAL SUMMARY ================\n")
    print(final_summary)
    
    return final_summary, combined_extractions

In [119]:
df = pd.DataFrame(list(filename_to_len.items()), columns=['Filename', 'Length'])

df.sort_values(by='Length', ascending=True, inplace=True)
df[df["Filename"] == "Putusan_PN_JAKARTA_BARAT_Nomor_467_Pdt.G_2023_PN_Jkt.Brt.pdf"]

,Filename,Length
1407,Putusan_PN_JAKARTA_BARAT_Nomor_467_Pdt.G_2023_...,323287


In [120]:
if __name__ == "__main__":
    ENVIRONMENT = "openai" 
    
    if ENVIRONMENT == "openai":
        active_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        active_model = "gpt-5.4-mini"
    else:
        active_client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
        active_model = "QuantTrio/Qwen3.5-27B-AWQ"

    response = parse_pdf(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\cleaned_downloads\Putusan_PN_JAKARTA_BARAT_Nomor_467_Pdt.G_2023_PN_Jkt.Brt.pdf", method="text")
    pages_list = [page.markdown for page in response.pages]
    
    final_output, worker_output = await run_summarization_pipeline(active_client, active_model, pages_list)

🚀 Starting Pipeline using model: gpt-5.4-mini...

📝 [Phase 1] Generating Blueprint...
✅ Blueprint Generated:
 - **1) Subjek & Objek**
  - **[Penggugat]**: AMRI
  - **[Tergugat]**:
    - [Tergugat I]: Ny. Leliana Widjaya
    - [Tergugat II]: PT. Suplaindo Sejahtera
    - [Tergugat III]: Hendra Sutanto
    - [Tergugat IV]: Kepala Kantor Pertanahan Kota Administrasi Jakarta Barat
  - **[Objek Sengketa]**:
    - Sebidang tanah garapan di Jl. Prepedan Dalam RT.001/RW.009, Kelurahan Kamal (dahulu Kel. Tegalalur), Kecamatan Kalideres, Jakarta Barat
    - Luas sekitar **3.538 m2**
    - Dikaitkan dengan **Girik No. 909 Persil 82.S.RR/I atas nama MENAK**
    - Dalam sengketa juga muncul **SHM No. 3494/Kamal** atas nama [Tergugat I]

- **2) Garis Besar Konflik**
  - Inti sengketa adalah **sengketa kepemilikan tanah**.
  - [Penggugat] mengklaim sebagai pemilik sah berdasarkan rangkaian pengoperan hak dari pihak sebelumnya.
  - [Tergugat I] menguasai tanah dan memiliki **SHM No. 3494/Kamal** yang 

In [117]:
print(worker_output)


--- CHUNK 1 EXTRACTIONS ---
### [A] Analisis Chunk #1
- **Kategori Dokumen**: Identitas Para Pihak dan Duduk Perkara
- **Detail Unik**:
  - Dokumen memuat identitas lengkap para pihak dalam perkara, yaitu [Penggugat] AMRI dan para [Tergugat] I sampai IV beserta alamat masing-masing.
  - Tercantum nomor perkara **467/Pdt.G/2023/PN.JKT.BRT** dan pengadilan yang memeriksa, yaitu Pengadilan Negeri Jakarta Barat.
  - Pada bagian duduk perkara, [Penggugat] menyatakan memperoleh tanah garapan melalui **Akta Pengoperan Hak No. 520 tanggal 31 Desember 2016** dari Ny. LILY.
  - [Objek Sengketa] dijelaskan berada di **Jl. Prepedan Dalam RT.001/RW.009, Kelurahan Kamal**, luas **± 3.538 m2**, dengan batas-batas tanah yang dirinci.
  - [Penggugat] juga menguraikan riwayat penguasaan tanah dari Ny. LILY, lalu ke JOSEPH SINAY sebagai kuasa dari LEE DARMAWAN KERTARAHARDJA HARIANTO alias LEE CHIN KIAT, dan seterusnya ke asal tanah dari MENAK.
  - [Penggugat] menuduh [Tergugat I] menguasai tanah berdasa